In [ ]:
# Parameters — modify these to analyze a different model/session
MODEL_ID      = "lgbm_solusdt_l_fw60_2101_2605"
DIRECTION     = "long"   # "long" or "short"
STRATEGY_DIR  = ""       # default: auto-derived from MODEL_ID
VALID_START   = ""       # default: from strategy_artifact fit_period.start
VALID_END     = ""       # default: from strategy_artifact fit_period.end


## How the Strategy Works

The strategy converts the model's raw prediction score into a trading decision using three offline-derived components:

1. **Rank calibration** -- the raw `short_pred` score is mapped to a rank percentile `short_pct` (0-1) using a lookup table built on the calibration period (`rank_lookup_short.parquet`). **Low `short_pct`** (near 0) means the current bar is in the top percentile of short signals — the most negative short_mfe predictions. See: `_doc_/methodology_doc/6100_strategy_calibration.md`

2. **Entry rule** -- a bar triggers a short entry when `(1 - short_pct) >= 0.94`, i.e. `short_pct <= 0.06` (bottom 6% of the calibration distribution = most extreme short signals). Entry is at the close price of the trigger bar. See: `_doc_/methodology_doc/6300_strategy_grid_search.md`

3. **Exit rule** -- two possible exits, whichever comes first:
   - **Take Profit**: when the next bar's **low** reaches `entry_close x exp(bucket_median_mfe)`, i.e. approximately -1.16% log return below entry (price drops to TP)
   - **Timeout**: if TP is not reached within 60 bars (60 minutes), the position closes at the 60th bar's close
   - No stop loss is set (`sl_spec: none`)

The TP level (`bucket_median_mfe = -0.011611`) is the median realized MFE across all top-bucket calibration samples. Short: profit when price falls.

See overall strategy design: `_doc_/methodology_doc/6000_strategy.md`

In [ ]:
import sys
import json
import base64
import io
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML

from analyst.lib.db_utils import find_repo_root, db_path as _live_db_path
from analyst.lib.plot_utils import CQ_COLORS, CQ_SEQUENCE, setup_cq_theme
setup_cq_theme()

REPO    = find_repo_root()
LIVE_DB = _live_db_path()

_strategy_dir = Path(STRATEGY_DIR) if STRATEGY_DIR else REPO / "artifacts" / MODEL_ID / "strategy"
_artifact     = json.loads((_strategy_dir / "strategy_artifact.json").read_text(encoding="utf-8"))
ENTRY_CUTOFF  = float(_artifact["decision_params"]["entry_cutoff"])

if not VALID_START:
    VALID_START = _artifact.get("fit_period", {}).get("start", "")
if not VALID_END:
    VALID_END = _artifact.get("fit_period", {}).get("end", "")

_lookup_file = f"rank_lookup_{DIRECTION}.parquet"
rl           = pd.read_parquet(_strategy_dir / _lookup_file)
_pred_col    = f"{DIRECTION}_pred"
_pct_col     = f"{DIRECTION}_pct"

if DIRECTION == "long":
    _top_bucket = rl[rl["score_pct"] >= ENTRY_CUTOFF]
    TP_LOG      = float(_top_bucket["bucket_median_mfe"].iloc[0])
else:
    _top_bucket = rl[rl["score_pct"] <= (1.0 - ENTRY_CUTOFF)]
    TP_LOG      = float(_top_bucket["bucket_median_mfe"].iloc[-1])
MAX_HOLD = int(_artifact["decision_params"]["max_hold_minutes"])

DOT_GREEN  = "#0ecb81"
DOT_YELLOW = "#f0b90b"
DOT_RED    = "#f6465d"


def simulate_outcome(trigger_time, entry_close, df_all):
    tp_price = entry_close * np.exp(TP_LOG)
    future   = df_all[df_all["open_time"] > trigger_time].iloc[:MAX_HOLD]
    if DIRECTION == "long":
        if (future["high"] >= tp_price).any():
            return DOT_GREEN, tp_price
        exit_close = future.iloc[-1]["close"] if len(future) == MAX_HOLD else entry_close
        return (DOT_YELLOW if exit_close > entry_close else DOT_RED), exit_close
    else:
        if (future["low"] <= tp_price).any():
            return DOT_GREEN, tp_price
        exit_close = future.iloc[-1]["close"] if len(future) == MAX_HOLD else entry_close
        return (DOT_YELLOW if exit_close < entry_close else DOT_RED), exit_close


_valid_start_dt = pd.Timestamp(VALID_START)
_chart_end      = (_valid_start_dt + pd.Timedelta(days=7)).strftime("%Y-%m-%d")

con = duckdb.connect(LIVE_DB, read_only=True)
df  = con.execute(f"""
    SELECT p.open_time, p.long_pred, p.short_pred, o.high, o.low, o.close
    FROM predictions p
    JOIN ohlcv o ON p.open_time = o.open_time
    WHERE p.open_time >= '{VALID_START}' AND p.open_time < '{_chart_end}'
    ORDER BY p.open_time
""").fetchdf()
con.close()

df["open_time"] = pd.to_datetime(df["open_time"])
df[_pct_col]    = np.interp(
    df[_pred_col].to_numpy(),
    rl["score_raw"].to_numpy(),
    rl["score_pct"].to_numpy(),
).clip(0, 1)

if DIRECTION == "long":
    triggers = df[df[_pct_col] >= ENTRY_CUTOFF]
else:
    triggers = df[(1.0 - df[_pct_col]) >= ENTRY_CUTOFF]

_entry_label = (
    f"{ENTRY_CUTOFF}"
    if DIRECTION == "long"
    else f"{ENTRY_CUTOFF}  (short: score_pct <= {1.0 - ENTRY_CUTOFF:.2f})"
)
display(pd.DataFrame([
    {"Key": "Rows loaded",    "Value": f"{len(df):,}"},
    {"Key": "Date range",     "Value": f"{df['open_time'].min().date()} - {df['open_time'].max().date()}"},
    {"Key": "Entry cutoff",   "Value": _entry_label},
    {"Key": "TP log return",  "Value": f"{TP_LOG:.6f}  ({TP_LOG*100:.4f}%)"},
    {"Key": "Max hold (min)", "Value": f"{MAX_HOLD}"},
    {"Key": "Total triggers", "Value": f"{len(triggers)}"},
]).set_index("Key"))


## Strategy Summary

In [ ]:
rows = [
    {"Parameter": "Model",             "Value": _artifact[f"{DIRECTION}_model"]},
    {"Parameter": "Direction",         "Value": DIRECTION},
    {"Parameter": "Entry cutoff",      "Value": ENTRY_CUTOFF},
    {"Parameter": "TP spec",           "Value": _artifact["decision_params"]["tp_spec"]},
    {"Parameter": "SL spec",           "Value": _artifact["decision_params"]["sl_spec"]},
    {"Parameter": "Max hold (min)",    "Value": _artifact["decision_params"]["max_hold_minutes"]},
    {"Parameter": "Signal mode",       "Value": _artifact["signal_mode"]},
    {"Parameter": "N trades (valid)",  "Value": _artifact["metrics"]["n_trades"]},
    {"Parameter": "Win rate",          "Value": f"{_artifact['metrics']['win_rate']:.1%}"},
    {"Parameter": "Compounded return", "Value": f"{_artifact['metrics']['compounded_return_pct']:.2f}%"},
    {"Parameter": "Fit period",        "Value": f"{_artifact['fit_period']['start']} - {_artifact['fit_period']['end']}"},
    {"Parameter": "Chart period",      "Value": f"{VALID_START} - {_chart_end} (first validation week)"},
]

summary_df = pd.DataFrame(rows)
summary_df.style.set_table_attributes('class="dataframe"').hide(axis="index")


## Weekly Trade Summary

In [ ]:
# Weekly trade summary — full validation period ($1000/trade sizing)
con2   = duckdb.connect(LIVE_DB, read_only=True)
df_all = con2.execute(f"""
    SELECT p.open_time, p.long_pred, p.short_pred, o.high, o.low, o.close
    FROM predictions p
    JOIN ohlcv o ON p.open_time = o.open_time
    WHERE p.open_time >= '{VALID_START}'
    ORDER BY p.open_time
""").fetchdf()
con2.close()

df_all["open_time"] = pd.to_datetime(df_all["open_time"])
df_all[_pct_col] = np.interp(
    df_all[_pred_col].to_numpy(),
    rl["score_raw"].to_numpy(),
    rl["score_pct"].to_numpy()
).clip(0, 1)

if DIRECTION == "long":
    all_trig = df_all[df_all[_pct_col] >= ENTRY_CUTOFF].copy()
else:
    all_trig = df_all[(1.0 - df_all[_pct_col]) >= ENTRY_CUTOFF].copy()

TRADE_SIZE   = 1000.0
outcome_list = []

for _, row in all_trig.iterrows():
    t0, ep = row["open_time"], row["close"]
    tp     = ep * np.exp(TP_LOG)
    future = df_all[df_all["open_time"] > t0].iloc[:MAX_HOLD]
    if DIRECTION == "long":
        if (future["high"] >= tp).any():
            pnl, outcome = TRADE_SIZE * (np.exp(TP_LOG) - 1), "green"
        else:
            ex  = future.iloc[-1]["close"] if len(future) == MAX_HOLD else ep
            pnl = TRADE_SIZE * (ex / ep - 1)
            outcome = "yellow" if pnl > 0 else "red"
    else:
        if (future["low"] <= tp).any():
            pnl, outcome = TRADE_SIZE * (1 - np.exp(TP_LOG)), "green"
        else:
            ex  = future.iloc[-1]["close"] if len(future) == MAX_HOLD else ep
            pnl = TRADE_SIZE * (1 - ex / ep)
            outcome = "yellow" if pnl > 0 else "red"
    outcome_list.append({"open_time": t0, "outcome": outcome, "pnl": pnl})

res_df         = pd.DataFrame(outcome_list)
res_df["week"] = res_df["open_time"].dt.to_period("W").dt.start_time.dt.date

weekly = res_df.groupby("week").apply(lambda g: pd.Series({
    "Trades":  len(g),
    "Green":   (g["outcome"] == "green").sum(),
    "Yellow":  (g["outcome"] == "yellow").sum(),
    "Red":     (g["outcome"] == "red").sum(),
    "G_gain":  g.loc[g["outcome"] == "green",  "pnl"].sum(),
    "Y_gain":  g.loc[g["outcome"] == "yellow", "pnl"].sum(),
    "R_gain":  g.loc[g["outcome"] == "red",    "pnl"].sum(),
}), include_groups=False).reset_index()

weekly["TP Rate"] = (weekly["Green"] / weekly["Trades"] * 100).round(1).astype(str) + "%"
weekly["Net ($)"] = weekly["G_gain"] + weekly["Y_gain"] + weekly["R_gain"]

totals = {
    "week": "TOTAL",
    "Trades":  int(weekly["Trades"].sum()),
    "Green":   int(weekly["Green"].sum()),
    "Yellow":  int(weekly["Yellow"].sum()),
    "Red":     int(weekly["Red"].sum()),
    "G_gain":  weekly["G_gain"].sum(),
    "Y_gain":  weekly["Y_gain"].sum(),
    "R_gain":  weekly["R_gain"].sum(),
    "TP Rate": f"{weekly['Green'].sum()/weekly['Trades'].sum()*100:.1f}%",
    "Net ($)": weekly["Net ($)"].sum(),
}

from IPython.display import Markdown


def fmt_usd(v):
    sign = "+" if v >= 0 else ""
    return f"{sign}{v:,.0f}"


header   = "| Week | Trades | Green | Yellow | Red | TP Rate | Green $ | Yellow $ | Red $ | Net $ |"
sep      = "| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"
md_lines = [header, sep]

for _, r in weekly.iterrows():
    md_lines.append(
        f"| {r['week']} | {int(r['Trades'])} "
        f"| <span style='color:#0ecb81;font-weight:bold'>{int(r['Green'])}</span> "
        f"| <span style='color:#f0b90b;font-weight:bold'>{int(r['Yellow'])}</span> "
        f"| <span style='color:#f6465d;font-weight:bold'>{int(r['Red'])}</span> "
        f"| {r['TP Rate']} "
        f"| <span style='color:#0ecb81'>{fmt_usd(r['G_gain'])}</span> "
        f"| <span style='color:#f0b90b'>{fmt_usd(r['Y_gain'])}</span> "
        f"| <span style='color:#f6465d'>{fmt_usd(r['R_gain'])}</span> "
        f"| **{fmt_usd(r['Net ($)'])}** |"
    )

md_lines.append(
    f"| **{totals['week']}** | **{totals['Trades']}** "
    f"| **<span style='color:#0ecb81'>{totals['Green']}</span>** "
    f"| **<span style='color:#f0b90b'>{totals['Yellow']}</span>** "
    f"| **<span style='color:#f6465d'>{totals['Red']}</span>** "
    f"| **{totals['TP Rate']}** "
    f"| **<span style='color:#0ecb81'>{fmt_usd(totals['G_gain'])}</span>** "
    f"| **<span style='color:#f0b90b'>{fmt_usd(totals['Y_gain'])}</span>** "
    f"| **<span style='color:#f6465d'>{fmt_usd(totals['R_gain'])}</span>** "
    f"| **{fmt_usd(totals['Net ($)'])}** |"
)

display(Markdown(chr(10).join(md_lines)))


## Validation Period - Day Charts

One section per week with at least one short trigger. Each section contains 7-day tabsets. Dot colors: green = TP hit (low touched), yellow = timeout positive (price dropped), red = timeout negative (price rose).

In [ ]:
#| echo: false
import base64, io
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 80, 'font.size': 9})

DOT_COLOR = {'green': '#0ecb81', 'yellow': '#f0b90b', 'red': '#f6465d'}

# Outcome lookup: trigger timestamp -> dot color
outcome_lookup = {
    pd.Timestamp(r['open_time']): DOT_COLOR[r['outcome']]
    for r in outcome_list
}

# Trigger weeks (weeks with at least 1 trade)
trigger_weeks = sorted(res_df['week'].unique())

accordion_items = []

for w_idx, week_date in enumerate(trigger_weeks):
    week_start = pd.Timestamp(week_date)
    week_end   = week_start + pd.Timedelta(days=6)
    week_str   = week_start.strftime('%Y-%m-%d')
    n_trig_week = int((res_df['week'] == week_date).sum())

    nav_items = []
    tab_panes = []
    first_tab = True

    for day_offset in range(7):
        day = week_start + pd.Timedelta(days=day_offset)
        day_str = day.strftime('%Y-%m-%d')
        tab_id  = f'tab-{week_str}-{day_offset}'

        day_df   = df_all[df_all['open_time'].dt.date == day.date()].copy()
        if DIRECTION == "long":
            triggers_day = day_df[day_df[_pct_col] >= ENTRY_CUTOFF]
        else:
            triggers_day = day_df[(1.0 - day_df[_pct_col]) >= ENTRY_CUTOFF]

        fig, (ax1, ax2) = plt.subplots(
            2, 1, sharex=True, figsize=(12, 4),
            gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.05}
        )

        if len(day_df) > 0:
            ax1.plot(day_df['open_time'], day_df['close'],
                     color='#2196F3', lw=0.7, zorder=2)
            ax2.plot(day_df['open_time'], day_df[_pct_col],
                     color='#2196F3', lw=0.8, zorder=2)
            _thr = ENTRY_CUTOFF if DIRECTION == "long" else 1.0 - ENTRY_CUTOFF
            ax2.axhline(_thr, color='#999', lw=0.8, ls='--', zorder=3)

            for _, trow in triggers_day.iterrows():
                color = outcome_lookup.get(pd.Timestamp(trow['open_time']), '#aaa')
                ax1.scatter([trow['open_time']], [trow['close']],
                            color=color, marker='o', s=60, zorder=5)

        ax2.set_ylim(-0.02, 1.02)
        ax2.set_ylabel(f'{_pct_col}')
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax2.xaxis.set_major_locator(mdates.HourLocator(interval=4))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
        ax2.set_xlabel('UTC')
        ax1.set_ylabel('Close (USDT)')
        ax1.tick_params(axis='x', labelbottom=False)

        _peak = (day_df[_pct_col].max() if DIRECTION == "long" else (1.0 - day_df[_pct_col]).max()) if len(day_df) else 0
        n_trig_day = len(triggers_day)
        fig.suptitle(
            f'{day_str}  triggers: {n_trig_day}  peak_signal: {_peak:.3f}',
            fontsize=9, y=1.01
        )
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode()
        plt.close(fig)

        active_cls = 'active' if first_tab else ''
        show_cls   = 'show active' if first_tab else ''
        nav_items.append(
            f'<li class="nav-item" role="presentation">'
            f'<button class="nav-link {active_cls}" id="{tab_id}-btn" '
            f'data-bs-toggle="tab" data-bs-target="#{tab_id}" '
            f'type="button" role="tab">{day.strftime("%a %d")}</button></li>'
        )
        tab_panes.append(
            f'<div class="tab-pane fade {show_cls}" id="{tab_id}" role="tabpanel">'
            f'<img src="data:image/png;base64,{img_b64}" '
            f'style="width:100%;max-width:1100px">'
            f'</div>'
        )
        first_tab = False

    tabset_html = (
        '<ul class="nav nav-tabs mb-2" role="tablist">' + ''.join(nav_items) + '</ul>'
        '<div class="tab-content mb-2">' + ''.join(tab_panes) + '</div>'
    )

    badge = f'<span class="badge bg-success ms-2">{n_trig_week} trigger</span>'
    is_first = (w_idx == 0)
    collapse_cls = 'show' if is_first else ''
    btn_cls = '' if is_first else ' collapsed'

    accordion_items.append(
        f'<div class="accordion-item">'
        f'<h2 class="accordion-header">'
        f'<button class="accordion-button{btn_cls}" type="button" '
        f'data-bs-toggle="collapse" data-bs-target="#week-{week_str}">'
        f'{week_str} &mdash; {week_end.strftime("%Y-%m-%d")}{badge}'
        f'</button></h2>'
        f'<div id="week-{week_str}" class="accordion-collapse collapse {collapse_cls}">'
        f'<div class="accordion-body p-2">{tabset_html}</div>'
        f'</div></div>'
    )

display(HTML(
    '<div class="accordion" id="weekAccordion">'
    + ''.join(accordion_items)
    + '</div>'
))
